In [4]:
import pandas as pd

# 1. Cargar el archivo CSV
df = pd.read_csv('seed_results.csv')

# Nombres exactos de las columnas en el DataFrame
col_rec = 'recovered_seed_col'
entry_rec = 'recovered_seed_entry'
col_real = 'col_real_concatenated_string'
entry_real = 'entry_real_concatenated_string'

# Asegurar tipo de dato string para la comparación
for col in [col_rec, entry_rec, col_real, entry_real]:
    df[col] = df[col].astype(str)

# 2. Función auxiliar para calcular métricas de coincidencia y diferencia
def compare_strings(recovered_str, real_str):
    total = len(real_str)
    if total == 0 or len(recovered_str) == 0:
        return 0.0, 0, total
    
    # Conteo de coincidencias y diferencias posición por posición
    diffs = sum(1 for a, b in zip(recovered_str, real_str) if a != b)
    matches = total - diffs
    accuracy_pct = (matches / total) * 100.0
    
    return accuracy_pct, diffs, total

# 3. Aplicar el cálculo a cada fila para Columnas Completas
col_metrics = df.apply(lambda r: compare_strings(r[col_rec], r[col_real]), axis=1)
df['col_accuracy_pct'] = [m[0] for m in col_metrics]
df['col_diff_count'] = [m[1] for m in col_metrics]
df['col_total_len'] = [m[2] for m in col_metrics]

# 4. Aplicar el cálculo a cada fila para Entradas de Bits
entry_metrics = df.apply(lambda r: compare_strings(r[entry_rec], r[entry_real]), axis=1)
df['entry_accuracy_pct'] = [m[0] for m in entry_metrics]
df['entry_diff_count'] = [m[1] for m in entry_metrics]
df['entry_total_len'] = [m[2] for m in entry_metrics]

# 5. Conteo de instancias con 100% de éxito (0 diferencias)
total_instances = len(df)
col_100_count = (df['col_diff_count'] == 0).sum()
entry_100_count = (df['entry_diff_count'] == 0).sum()

print("=== RESULTADOS DE COMPARACIÓN Y DIFERENCIAS EN SEMILLAS ===")
print(f"Total de instancias analizadas: {total_instances}\n")

print("1. Semilla por Columnas Completas (recovered_seed_col vs real):")
print(f"   -> Precisión promedio: {df['col_accuracy_pct'].mean():.2f}%")
print(f"   -> Promedio de bits diferentes: {df['col_diff_count'].mean():.2f} / {df['col_total_len'].iloc[0]}")
print(f"   -> Instancias con 0 diferencias (100% éxito): {col_100_count} / {total_instances} ({col_100_count / total_instances * 100:.1f}%)\n")

print("2. Semilla por Entradas de Bits (recovered_seed_entry vs real):")
print(f"   -> Precisión promedio: {df['entry_accuracy_pct'].mean():.2f}%")
print(f"   -> Promedio de bits diferentes: {df['entry_diff_count'].mean():.2f} / {df['entry_total_len'].iloc[0]}")
print(f"   -> Instancias con 0 diferencias (100% éxito): {entry_100_count} / {total_instances} ({entry_100_count / total_instances * 100:.1f}%)\n")

# 6. Agrupación y resumen por valor de Beta
print("=== RESUMEN DE DIFERENCIAS PROMEDIO SEGÚN BETA ===")
summary = df.groupby('beta').agg(
    col_pct_mean=('col_accuracy_pct', 'mean'),
    col_avg_diffs=('col_diff_count', 'mean'),
    col_total_bits=('col_total_len', 'first'),
    col_exact_100=('col_diff_count', lambda x: (x == 0).sum()),
    entry_pct_mean=('entry_accuracy_pct', 'mean'),
    entry_avg_diffs=('entry_diff_count', 'mean'),
    entry_total_bits=('entry_total_len', 'first'),
    entry_exact_100=('entry_diff_count', lambda x: (x == 0).sum()),
    total_samples=('instance_id', 'count')
)

summary

=== RESULTADOS DE COMPARACIÓN Y DIFERENCIAS EN SEMILLAS ===
Total de instancias analizadas: 250

1. Semilla por Columnas Completas (recovered_seed_col vs real):
   -> Precisión promedio: 93.55%
   -> Promedio de bits diferentes: 16.00 / 248
   -> Instancias con 0 diferencias (100% éxito): 0 / 250 (0.0%)

2. Semilla por Entradas de Bits (recovered_seed_entry vs real):
   -> Precisión promedio: 91.65%
   -> Promedio de bits diferentes: 21.38 / 256
   -> Instancias con 0 diferencias (100% éxito): 7 / 250 (2.8%)

=== RESUMEN DE DIFERENCIAS PROMEDIO SEGÚN BETA ===


,col_pct_mean,col_avg_diffs,col_total_bits,col_exact_100,entry_pct_mean,entry_avg_diffs,entry_total_bits,entry_exact_100,total_samples
beta,,,,,,,,,
0.03,98.250000,4.34,248,0,98.929688,2.74,256,3,50
0.05,97.290323,6.72,248,0,98.937500,2.72,256,4,50
0.10,93.548387,16.00,248,0,90.515625,24.28,256,0,50
0.15,90.645161,23.20,248,0,86.906250,33.52,256,0,50
0.20,88.008065,29.74,248,0,82.945312,43.66,256,0,50


In [3]:
summary

,col_pct_mean,col_exact_100,entry_pct_mean,entry_exact_100,total_samples
beta,,,,,
0.03,98.250000,0,98.929688,3,50
0.05,97.290323,0,98.937500,4,50
0.10,93.548387,0,90.515625,0,50
0.15,90.645161,0,86.906250,0,50
0.20,88.008065,0,82.945312,0,50
